In [ ]:
import polars as pl
import pandas as pd
import plotly.express as px

In [79]:
# Define pathings
data_path = '../../03_output/01_enriched_results/'
out_tables_path = '../../03_output/02_result_tables/'

In [3]:
# Define list of medieval keywords. Note that st and th verbs are all medieval.
medieval_pronouns = ['mine', 'mineself', 'thee', 'thine', 'thou', 'thy', 'thyself', 'ye']

medieval_modal_verbs = ['couldst', 'shalt', 'shouldst', 'wilt']

medieval_auxiliary_verbs = ['art']

In [4]:
# Load results and filter for medieval keywords. Also add a column to mark them as either verbs or pronouns.
medieval_pronouns_df = pl.read_csv(data_path + 'pronoun_analysis.csv', separator=';')\
                         .filter(pl.col('KWIC').is_in(medieval_pronouns))\
                         .with_columns(pl.lit('pronoun').alias('type'))

medieval_modal_verbs_df = pl.read_csv(data_path + 'modal_verbs_analysis.csv', separator=';')\
                            .filter(pl.col('KWIC').is_in(medieval_modal_verbs))\
                            .with_columns(pl.lit('verb').alias('type'))

medieval_auxiliary_verbs_df = pl.read_csv(data_path + 'auxiliary_verbs_analysis.csv', separator=';')\
                                .filter(pl.col('KWIC').is_in(medieval_auxiliary_verbs))\
                                .with_columns(pl.lit('verb').alias('type'))

st_verbs_df = pl.read_csv(data_path + 'st_verbs_analysis.csv', separator=';')\
                .with_columns(pl.lit('verb').alias('type'))

th_verbs_df = pl.read_csv(data_path + 'th_verbs_analysis.csv', separator=';')\
                .with_columns(pl.lit('verb').alias('type'))



In [ ]:
# Concatenate all results into a single dataframe for ease of use. 
# Also unify class and age into a single column for ease of analysis
medieval_kw_df = pl.concat([medieval_pronouns_df, medieval_modal_verbs_df, 
                            medieval_auxiliary_verbs_df, st_verbs_df, th_verbs_df])\
                   .with_columns((pl.col('Class') + pl.lit('-') + pl.col('Age')).alias('Class-Age'))\
                   .with_columns(pl.when(pl.col('Class-Age').is_null())
                                 .then(pl.lit('Narrator'))
                                 .otherwise(pl.col('Class-Age'))
                                 .alias('Class-Age'))\

medieval_kw_df

Left,KWIC,Right,Character,Class,Age,type,Class-Age
str,str,str,str,str,str,str,str
""", where they are locked away, …","""thou""","""art new. Thou fared well to …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old""","""pronoun""","""High-Old"""
""" locked away, to await the end…","""thou""","""fared well to find me. But c…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old""","""pronoun""","""High-Old"""
"""ld. ...This is your fate. On…","""thee""","""not for the grave of Sir Artor…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old""","""pronoun""","""High-Old"""
"""Lords, Lordran. ALVINA OF TH…","""thine""","""own respect, go not yonder kno…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old""","""pronoun""","""High-Old"""
"""ared well to find me. But co…","""thou""","""art a strange one! Neverthel…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old""","""pronoun""","""High-Old"""
…,…,…,…,…,…,…,…
"""he Darkmoon trespasseth upon t…","""endeth""",""" the Godmother, and now thou s…","""DARK SUN GWYNDOLIN""","""High""","""Old""","""verb""","""High-Old"""
"""sen Undead. Come hither, child…","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT""","""High""","""Old""","""verb""","""High-Old"""
"""ire of our world. A grave and …","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT""","""High""","""Old""","""verb""","""High-Old"""


### Frequency of Medieval forms

In [83]:
# Construct table of frequencies for all medieval forms.
medieval_kw_freq_df = medieval_kw_df.group_by('KWIC').agg(pl.first('type'), pl.len().alias('freq')).sort('freq', descending= True)\
                #.with_columns((100*pl.col('freq')/pl.sum('freq')).round(2).alias('freq_%'))

medieval_kw_freq_df

KWIC,type,freq
str,str,u32
"""thou""","""pronoun""",92
"""thee""","""pronoun""",70
"""thine""","""pronoun""",38
"""thy""","""pronoun""",20
"""art""","""verb""",19
…,…,…
"""leavest""","""verb""",1
"""clingeth""","""verb""",1
"""seekest""","""verb""",1


In [84]:
# Construct table of frequencies for all medieval verbs.
medieval_verbs_freq_df = medieval_kw_df.filter(pl.col('type') == 'verb').group_by('KWIC').agg(pl.len().alias('freq')).sort('freq', descending= True)\
                # .with_columns((100*pl.col('freq')/pl.sum('freq')).round(2).alias('freq_%'))

medieval_verbs_freq_df.write_csv(out_tables_path + 'medieval_verbs_freq.csv', separator=';')

medieval_verbs_freq_df

KWIC,freq
str,u32
"""hast""",19
"""art""",19
"""shalt""",15
"""cometh""",8
"""dost""",7
…,…
"""commiteth""",1
"""needst""",1
"""clingeth""",1


In [85]:
# Construct table of frequencies for all medieval pronouns.
medieval_pronouns_freq_df = medieval_kw_df.filter(pl.col('type') == 'pronoun')\
                                    .group_by('KWIC').agg(pl.len().alias('freq')).sort('freq', descending= True)\
                                    # .with_columns((100*pl.col('freq')/pl.sum('freq')).round(2).alias('freq_%'))

medieval_pronouns_freq_df.write_csv(out_tables_path + 'medieval_pronouns_freq.csv', separator=';')

medieval_pronouns_freq_df

KWIC,freq
str,u32
"""thou""",92
"""thee""",70
"""thine""",38
"""thy""",20
"""mine""",16
"""mineself""",4
"""thyself""",4
"""ye""",3


In [ ]:
# Get list of top 10 verbs and pronouns in terms of frequency for later graphs
top_10_freq_verbs = medieval_verbs_freq_df.head(10)['KWIC'].to_list()

top_10_freq_kw = medieval_kw_freq_df.head(10)['KWIC'].to_list()


### Medieval forms by class

In [91]:
# Create aggregated table by KWIC and Class-Age for frequency of all medieval forms
med_kw_by_class = medieval_kw_df.group_by(['KWIC', 'type', 'Class-Age']).agg(pl.len().alias('freq')).sort('freq', descending= True)\
                                # .with_columns((100*pl.col('freq')/pl.sum('freq')).round(2).alias('freq_%'))

med_kw_by_class.write_csv(out_tables_path + 'medieval_kw_by_class_freq.csv', separator=';')

med_kw_by_class

KWIC,type,Class-Age,freq
str,str,str,u32
"""thou""","""pronoun""","""High-Old""",89
"""thee""","""pronoun""","""High-Old""",67
"""thine""","""pronoun""","""High-Old""",37
"""hast""","""verb""","""High-Old""",19
"""thy""","""pronoun""","""High-Old""",19
…,…,…,…
"""commiteth""","""verb""","""High-Old""",1
"""tarnisheth""","""verb""","""High-Old""",1
"""leavest""","""verb""","""High-Old""",1


In [89]:
# Define color map for Class-Age for consistency.
color_map = {'High-Old':"#636efa",'Low-Young':'#ef553b','Low-Old':'#00cc96','Narrator':'#9b81cf'}

In [90]:
# Represent the top 10 medieval forms by frequency and Class-Age
med_kw_by_class_top_10 = med_kw_by_class.filter(pl.col('KWIC').is_in(top_10_freq_kw))

# Get a list with the proper total freq of each KWIC ordered in a descending manner.
kw_order_desc = med_kw_by_class_top_10.group_by('KWIC').agg(pl.sum('freq')).sort('freq', descending= True)['KWIC'].to_list()

fig = px.bar(med_kw_by_class_top_10.to_pandas(), y='freq', x='KWIC', color='Class-Age',
             color_discrete_map=color_map, category_orders= {'KWIC':kw_order_desc},
             title='Frequency of usage of neomedieval forms by class-age',
             width= 800, height= 600)
fig.show()

In [77]:
# Represent the top 10 medieval verbs by frequency and Class-Age
med_verbs_by_class_top_10 = med_kw_by_class.filter(pl.col('KWIC').is_in(top_10_freq_verbs))

# Get a list with the proper total freq of each KWIC ordered in a descending manner.
kw_order_desc = med_verbs_by_class_top_10.group_by('KWIC').agg(pl.sum('freq')).sort('freq', descending= True)['KWIC'].to_list()

fig = px.bar(med_verbs_by_class_top_10.to_pandas(), y='freq', x='KWIC', color='Class-Age',
             color_discrete_map=color_map, category_orders= {'KWIC':kw_order_desc},
             title='Frequency of usage of neomedieval verbs by class-age',
             width= 800, height= 600)
fig.show()

In [92]:
# Generate the aggregated frequency table by class-age
class_freq_df = med_kw_by_class.group_by('Class-Age').agg(pl.sum('freq')).sort('freq', descending= True)\
                               .with_columns((100*pl.col('freq')/pl.sum('freq')).round(2).alias('freq_%'))

class_freq_df.write_csv(out_tables_path + 'total_class_age_freq.csv', separator=';')

class_freq_df

Class-Age,freq,freq_%
str,u32,f64
"""High-Old""",345,94.26
"""Low-Young""",16,4.37
"""Low-Old""",3,0.82
"""Narrator""",2,0.55


In [78]:
# Represent the previous table
fig = px.bar(class_freq_df.to_pandas(), y='freq', x='Class-Age', color='Class-Age',
             color_discrete_map=color_map, title='Total frequency of usage of neomedieval forms by class-age',
             width= 800, height= 600)
fig.show()